# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [7]:
my_lane = "Refresh / Content Opportunity Scoring"

why_this_lane = """
I'm choosing the Refresh / Content Opportunity Scoring lane. SEO teams often
have limited time and resources, so knowing which underperforming pages to
review first is a real, practical decision, not just a descriptive report.
Specifically: for a content/SEO editor deciding which pages to fix first,
this lane produces a ranked priority queue with scores and reason codes,
so the editor can act directly on the top items rather than manually
auditing every page. A wrong call has a real cost, wasted editor hours on
a false alarm, or a genuinely declining page going unnoticed longer than it
should. A plain rule isn't enough here because the signals that indicate a
page needs attention interact in messy, shifting ways across different
clients and content types, real enough to learn, too tangled to hand-write
reliably. This also maps closely to work I've already done in my MSc
capstone project, a RAG-based Knowledge Graph QA system, where I built a
hop-aware pipeline that scores and routes queries by complexity and
evaluated results using ranking metrics like Hit@k and MRR - this lane
follows a similar structure, scoring and ranking content pages instead of
queries, so I can apply skills I already have while working with real
search/analytics data instead of academic benchmarks.
"""

print(my_lane)
print(why_this_lane)

Refresh / Content Opportunity Scoring

I'm choosing the Refresh / Content Opportunity Scoring lane. SEO teams often
have limited time and resources, so knowing which underperforming pages to
review first is a real, practical decision, not just a descriptive report.
Specifically: for a content/SEO editor deciding which pages to fix first,
this lane produces a ranked priority queue with scores and reason codes,
so the editor can act directly on the top items rather than manually
auditing every page. A wrong call has a real cost, wasted editor hours on
a false alarm, or a genuinely declining page going unnoticed longer than it
should. A plain rule isn't enough here because the signals that indicate a
page needs attention interact in messy, shifting ways across different
clients and content types, real enough to learn, too tangled to hand-write
reliably. This also maps closely to work I've already done in my MSc
capstone project, a RAG-based Knowledge Graph QA system, where I built a
hop-a

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [8]:
decision_action_cost = """
Decision: Which underperforming content pages a content/SEO editor should
review and fix first, given limited review time - not simply predicting
which pages will decline.

Who acts on it: A content editor or SEO team member, who works down the
ranked priority queue and applies fixes (schema updates, content rewrites,
internal linking) to the top-scored pages first.

Cost of a wrong recommendation: A false positive wastes editor hours
reviewing a page that wasn't actually a priority. A false negative means a
genuinely declining page is missed and keeps losing traffic and rankings
unnoticed for longer, which is the more expensive error since it delays
real revenue-impacting action.
"""

print(decision_action_cost)


Decision: Which underperforming content pages a content/SEO editor should
review and fix first, given limited review time - not simply predicting
which pages will decline.

Who acts on it: A content editor or SEO team member, who works down the
ranked priority queue and applies fixes (schema updates, content rewrites,
internal linking) to the top-scored pages first.

Cost of a wrong recommendation: A false positive wastes editor hours
reviewing a page that wasn't actually a priority. A false negative means a
genuinely declining page is missed and keeps losing traffic and rankings
unnoticed for longer, which is the more expensive error since it delays
real revenue-impacting action.



## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:
!git clone https://github.com/KarthikaAkkappilly/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 98 (delta 19), reused 79 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 1.84 MiB | 26.16 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/flyrank-ml-internship/flyrank-ml-internship


In [10]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

print(df['trend_direction'].value_counts(normalize=True) * 100)

no_position = (df['avg_position'] == 0).sum()
print(f"{no_position} pages ({no_position/len(df)*100:.1f}%) have no position data")

declining = df[df['trend_direction'] == 'down']
print(f"Median decline: {declining['trend_pct'].median():.1f}%")
print(f"Worst 10% decline threshold: {declining['trend_pct'].quantile(0.1):.1f}%")

(30000, 44)
trend_direction
down      54.206667
stable    19.873333
up        14.626667
new        7.453333
flat       3.840000
Name: proportion, dtype: float64
1205 pages (4.0%) have no position data
Median decline: -55.6%
Worst 10% decline threshold: -96.0%


In [11]:
takeaway = """
Across the 30,000-page starter dataset, 54.2% of pages are currently
trending down (trend_direction == 'down'), meaning decline is the majority
state, not an edge case. Among those declining pages, the median drop is
-55.6%, and the worst 10% have fallen 96% or more - a small manual review
process cannot keep up with a problem this large and this severe. This is
exactly the kind of prioritization problem a ranked scoring model is built
for: with over half the content base declining to varying degrees, editors
need a way to triage which pages are the most urgent 55%-drop cases versus
the catastrophic 96%-drop cases, rather than reviewing pages in an
arbitrary or ad hoc order.
"""
print(takeaway)


Across the 30,000-page starter dataset, 54.2% of pages are currently
trending down (trend_direction == 'down'), meaning decline is the majority
state, not an edge case. Among those declining pages, the median drop is
-55.6%, and the worst 10% have fallen 96% or more - a small manual review
process cannot keep up with a problem this large and this severe. This is
exactly the kind of prioritization problem a ranked scoring model is built
for: with over half the content base declining to varying degrees, editors
need a way to triage which pages are the most urgent 55%-drop cases versus
the catastrophic 96%-drop cases, rather than reviewing pages in an
arbitrary or ad hoc order.



## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [12]:
careful_words = """
What this work CAN say:

Observed: trend_direction and trend_pct describe what has already happened
to a page's performance over the measured window - these are historical
facts, not predictions.

Directional: the model's predicted decline percentage indicates which
pages are likely, based on measured pre-decline signals, to need review
soonest - a directional estimate for prioritization, not a guaranteed
number. A predicted -60% does not mean the page will decline by exactly
60%; it means the model's learned pattern points toward severe decline.

Decision-support: the resulting ranked queue is a recommendation to help
an editor decide where to spend limited review time first. The editor
still makes the final call on each page; the model narrows and orders
the list, it doesn't replace judgment.

What this work will NEVER say:

It will not claim causal proof. A high predicted decline does not mean we
know WHY a page is declining, only that its measured signals resemble
those of pages that have declined before. Correlation in the training
data is not evidence of a causal mechanism, and no experiment or
controlled test was run to establish cause.

It will not claim to predict Google's algorithm or reverse-engineer how
search ranking works. The model learns patterns in FlyRank's own
historical performance data, not Google's internal ranking logic.

It will not claim certainty about any individual page's future. All
predictions are probabilistic estimates based on historical patterns in
similar pages, not guarantees of what will happen to a specific page next.
"""
print(careful_words)


What this work CAN say:

Observed: trend_direction and trend_pct describe what has already happened
to a page's performance over the measured window - these are historical
facts, not predictions.

Directional: the model's predicted decline percentage indicates which
pages are likely, based on measured pre-decline signals, to need review
soonest - a directional estimate for prioritization, not a guaranteed
number. A predicted -60% does not mean the page will decline by exactly
60%; it means the model's learned pattern points toward severe decline.

Decision-support: the resulting ranked queue is a recommendation to help
an editor decide where to spend limited review time first. The editor
still makes the final call on each page; the model narrows and orders
the list, it doesn't replace judgment.

What this work will NEVER say:

It will not claim causal proof. A high predicted decline does not mean we
know WHY a page is declining, only that its measured signals resemble
those of pages t

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.